# GS55 — Step 2: Build ConvNeXt-StarDist tile dataset

Builds a balanced 256×256 tile dataset from the GS55 whole-slide images
and CODA-labelled GeoJSONs produced by Steps 0–1.

**Pipeline:**
1. Load per-slide cell counts → compute 4-tier sampling weights
2. Build slide manifest (NDPI + GeoJSON pairs)
3. Parse GeoJSON centroids, class labels, **and full polygon features**
4. Generate non-overlapping tile grid, filter low-cell tiles
5. Weighted unique tile selection (~40k tiles)
6. Slide-level train/val/test split
7. Oversample rare classes (train only, augmented `_dup{k}` copies)
8. Extract PNG tiles, per-tile GeoJSONs, uint16 instance masks, and inst2class.json sidecars
9. Write split CSVs and ConvNeXt-StarDist training config

**Output per tile** (under `dataset_256_40k_GS55/{train|val|test}/`):

| Folder | File | Content |
|--------|------|---------|
| `images/` | `tile_id.png` | RGB image (256×256) |
| `geojsons/` | `tile_id.geojson` | Localized polygon GeoJSON with class labels (for QuPath inspection) |
| `labels/` | `tile_id.png` | uint16 instance mask (0 = bg, 1…N = nuclei) |
| `labels/` | `tile_id_inst2class.json` | `{"1": "kidney", "2": "bone", ...}` |

| Cell | Purpose |
|------|---------|
| 1 | Imports |
| **2** | **Parameters ← edit here** |
| 3 | Class distribution + sampling weights |
| 4 | Build manifest |
| 5 | Parse GeoJSON annotations |
| 6 | Generate tile candidates |
| 7 | Weighted tile selection |
| 8 | Train / val / test split |
| 9 | Oversample rare classes |
| 10 | Extract tiles: PNG + GeoJSON + instance mask + inst2class.json |
| 11 | Write splits and training config |

---
## 1 · Imports

In [1]:
import json
import re
import sys
import yaml
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import openslide
from tqdm.auto import tqdm

sys.path.insert(0, str(Path.cwd()))
from dataset_utils import (
    calculate_hybrid_weights,
    get_slide_mpp,
    assign_cells_to_tiles,
    augment_image,
    augment_coords,
    augment_polygon_ring,
    clip_features_to_tile,
    rasterize_tile_features,
    write_tile_geojson,
    polygon_centroid,
)

print("Imports OK")

Imports OK


---
## 2 · Parameters  ← edit here

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────────
CELL_COUNTS_CSV = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\cell_type_analysis\cell_counts_per_slide.csv")
GEOJSON_DIR     = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\geojson_CODAclass")
# TODO: update NDPI_BASE_DIR to the folder containing GS55 whole-slide .ndpi files
NDPI_BASE_DIR   = Path(r"\\kittyserverdw\Andre_kit\data\monkey_fetus\bissected_monkey_GS55")
OUT_DIR         = Path(r"\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training")
OUT_BASE        = OUT_DIR / "dataset_256_50k_2testslides_GS55"

# ── Class palette ──────────────────────────────────────────────────────────────
LABELS = [
    "bone",   "brain",  "eye",       "heart",     "lungs",
    "GI",     "liver",  "spleen",    "pancreas",  "kidney",
    "mesokidney", "collagen", "ear", "nontissue", "thymus",
    "thyroid", "bladder", "skull",   "spleen2",
]
COLORS = [
    [214,212,161], [247,184, 67], [136,232, 95], [140, 13, 13], [ 38, 27,166],
    [ 13,125, 11], [179, 50,108], [228,235,131], [156, 96,235], [ 46,190,230],
    [150,255,245], [254,222,255], [235,154,108], [255,255,255], [  9, 64,116],
    [255,255, 74], [178,178,  0], [214,212,161], [ 54, 83, 89],
]

# ── Tile extraction ────────────────────────────────────────────────────────────
TILE_SIZE       = 256   # px at 20x
STRIDE          = 256   # non-overlapping grid
MIN_CELLS_TILE  = 5     # discard tiles with fewer cells than this
NONTISSUE_FRAC  = 0.75  # discard tiles where >75% of cells are "nontissue"
TARGET_TILES    = 50000 # unique tiles to sample before oversampling
SEED            = 1337

# ── Train / val / test split ───────────────────────────────────────────────────
TRAIN_RATIO     = 0.70
VAL_RATIO       = 0.20
MAX_TEST_SLIDES = 2     # cap test split to exactly this many slides
# TEST_RATIO  = 1 - TRAIN_RATIO - VAL_RATIO (computed automatically)

name_to_id = {name: i for i, name in enumerate(LABELS)}
id_to_name = {i: name for i, name in enumerate(LABELS)}

print(f"Cell counts CSV : {CELL_COUNTS_CSV.exists()}")
print(f"GeoJSON dir     : {GEOJSON_DIR.exists()}")
print(f"NDPI base dir   : {NDPI_BASE_DIR.exists()}")
print(f"Output base     : {OUT_BASE}")


Cell counts CSV : True
GeoJSON dir     : True
NDPI base dir   : True
Output base     : \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_40k_GS55


---
## 3 · Class distribution and sampling weights

In [3]:
# ── Load class distribution and compute sampling weights ───────────────────────
df_counts   = pd.read_csv(CELL_COUNTS_CSV, index_col=0)
if "TOTAL" in df_counts.columns:
    df_counts = df_counts.drop(columns=["TOTAL"])

class_totals   = df_counts.sum(axis=0).sort_values(ascending=False)
total_cells    = int(class_totals.sum())
hybrid_weights = calculate_hybrid_weights(class_totals)

print(f"Total cells: {total_cells:,}  |  Classes: {len(class_totals)}")
print(f"\n{'Class':<20} {'Count':>12} {'%':>7} {'Weight':>9}")
print("-" * 55)
for cls, cnt in class_totals.items():
    print(f"{cls:<20} {cnt:>12,} {cnt/total_cells*100:>6.1f}% {hybrid_weights[cls]:>9.3f}")

Total cells: 19,389,206  |  Classes: 19

Class                       Count       %    Weight
-------------------------------------------------------
collagen                8,528,692   44.0%     0.050
brain                   4,282,881   22.1%     0.050
liver                   2,183,804   11.3%     0.100
bone                    1,470,258    7.6%     0.500
nontissue                 786,130    4.1%     0.500
lungs                     534,154    2.8%     0.700
skull                     476,917    2.5%     0.784
GI                        383,685    2.0%     0.975
heart                     178,606    0.9%     2.093
eye                       137,996    0.7%     2.710
kidney                    111,204    0.6%     3.362
spleen                     73,773    0.4%     5.068
thymus                     72,540    0.4%     5.155
mesokidney                 50,315    0.3%     7.431
thyroid                    46,389    0.2%     8.060
ear                        28,860    0.1%    10.000
pancreas           

---
## 4 · Build slide manifest

In [4]:
# ── Build slide manifest (GeoJSON + NDPI path pairs) ──────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)

gj_files   = sorted(GEOJSON_DIR.glob("*.geojson"))
manifest   = []
no_ndpi    = []

for gj_path in gj_files:
    ndpi_path = NDPI_BASE_DIR / f"{gj_path.stem}.ndpi"
    if ndpi_path.exists():
        manifest.append({
            "slide_id":     gj_path.stem,
            "image_path":   str(ndpi_path),
            "geojson_path": str(gj_path),
        })
    else:
        no_ndpi.append(gj_path.stem)

manifest_path = OUT_DIR / "GS55_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Matched   : {len(manifest)} / {len(gj_files)} slides")
print(f"No NDPI   : {len(no_ndpi)}{' — check NDPI_BASE_DIR' if no_ndpi else ''}")
print(f"Manifest  → {manifest_path}")
if no_ndpi[:5]:
    print("  Missing:", no_ndpi[:5])

Matched   : 14 / 14 slides
No NDPI   : 0
Manifest  → \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\GS55_manifest.json


---
## 5 · Parse GeoJSON annotations

In [5]:
# ── Parse GeoJSON annotations → per-slide cell arrays ─────────────────────────
# slide_data[slide_id] stores:
#   cells_xy     : (N, 2) float32 centroids in slide pixel space
#   cells_labels : (N,)   int32   class indices
#   features     : list of N GeoJSON feature dicts, same order as cells_xy
#                  (used later to write per-tile GeoJSONs and rasterize masks)
slide_data = {}

for entry in tqdm(manifest, desc="Parsing GeoJSONs"):
    gj_path  = Path(entry["geojson_path"])
    raw      = json.loads(gj_path.read_text(encoding="utf-8"))
    all_feats = raw if isinstance(raw, list) else raw.get("features", [])

    centroids, labels, features = [], [], []
    for feat in all_feats:
        class_name = feat.get("properties", {}).get("classification", {}).get("name")
        if not class_name or class_name not in name_to_id:
            continue
        geom = feat.get("geometry", {})
        if geom.get("type") != "Polygon":
            continue
        ring = geom.get("coordinates", [[]])[0]
        cx, cy = polygon_centroid(ring)
        if cx is None:
            continue
        centroids.append([cx, cy])
        labels.append(name_to_id[class_name])
        features.append(feat)      # keep the full feature dict (same index as centroid)

    if centroids:
        slide_data[entry["slide_id"]] = {
            "image_path":   entry["image_path"],
            "cells_xy":     np.array(centroids, dtype=np.float32),
            "cells_labels": np.array(labels,    dtype=np.int32),
            "features":     features,   # list of GeoJSON feature dicts, aligned with cells_xy
        }

total_parsed = sum(len(v["cells_xy"]) for v in slide_data.values())
print(f"Parsed {len(slide_data)} slides, {total_parsed:,} cells")

Parsing GeoJSONs:   0%|          | 0/14 [00:00<?, ?it/s]

Parsed 14 slides, 19,389,206 cells


---
## 6 · Generate tile candidates

In [6]:
# ── Generate tile candidates ───────────────────────────────────────────────────
NONTISSUE_ID = name_to_id.get("nontissue", -1)

all_tiles = []
for slide_id, data in tqdm(slide_data.items(), desc="Building tile grid"):
    cells_xy     = data["cells_xy"]
    cells_labels = data["cells_labels"]
    tile_groups  = assign_cells_to_tiles(cells_xy, TILE_SIZE, STRIDE)

    for (tx, ty), cell_idxs in tile_groups.items():
        if len(cell_idxs) < MIN_CELLS_TILE:
            continue
        nt_frac = sum(1 for ci in cell_idxs if cells_labels[ci] == NONTISSUE_ID) / len(cell_idxs)
        if nt_frac > NONTISSUE_FRAC:
            continue
        class_counts = Counter(cells_labels[cell_idxs].tolist())
        all_tiles.append({
            "slide_id":    slide_id,
            "tile_x":      tx,
            "tile_y":      ty,
            "cell_indices": cell_idxs,
            "num_cells":   len(cell_idxs),
            "class_counts": class_counts,
        })

print(f"Candidate tiles: {len(all_tiles):,}")

Building tile grid:   0%|          | 0/14 [00:00<?, ?it/s]

Candidate tiles: 258,626


---
## 7 · Weighted unique tile selection

In [7]:
# ── Weighted unique tile selection ────────────────────────────────────────────
BIG5_NAMES = ["collagen", "bone", "brain", "liver", "nontissue"]
BIG5_IDS   = {name_to_id[n] for n in BIG5_NAMES if n in name_to_id}
RARE_IDS   = {
    name_to_id[n] for n in LABELS
    if n in class_totals.index and class_totals[n] / total_cells < 0.01
}

weights = []
valid   = []
for idx, tile in enumerate(all_tiles):
    tc = tile["num_cells"]
    if tc == 0:
        continue
    value = 0.0
    for cls_id, cnt in tile["class_counts"].items():
        cname = id_to_name.get(cls_id, "")
        if cls_id in BIG5_IDS:
            value += cnt * (0.01 if cname == "collagen" else 0.02 if cname == "nontissue" else 0.03)
        elif cls_id in RARE_IDS:
            value += cnt * 10.0
        else:
            value += cnt * 1.0
    value /= tc
    b5 = sum(tile["class_counts"].get(b, 0) for b in BIG5_IDS)
    if b5 / tc > 0.6:   value *= 0.01
    elif b5 / tc > 0.4: value *= 0.05
    valid.append(idx)
    weights.append(value)

weights = np.array(weights, dtype=float)
probs   = np.maximum(weights, 1e-10)
probs  /= probs.sum()

np.random.seed(SEED)
n_sample       = min(TARGET_TILES, len(valid))
sel_indices    = np.random.choice(valid, size=n_sample, replace=False, p=probs)
selected_tiles = [all_tiles[i] for i in sel_indices]

print(f"Selected {len(selected_tiles):,} unique tiles (target: {TARGET_TILES:,})")

# ── Class distribution after selection ────────────────────────────────────────
sampled_cc = Counter()
for tile in selected_tiles:
    sampled_cc.update(tile["class_counts"])

print(f"\n{'Class':<18} {'Original':>10} {'Sampled':>10} {'Ratio':>8}")
print("-" * 50)
for cname in LABELS:
    cid  = name_to_id[cname]
    orig = class_totals.get(cname, 0)
    samp = sampled_cc.get(cid, 0)
    print(f"{cname:<18} {orig:>10,} {samp:>10,} {samp/orig if orig else 0:>7.2f}x")

Selected 40,000 unique tiles (target: 40,000)

Class                Original    Sampled    Ratio
--------------------------------------------------
bone                1,470,258     62,638    0.04x
brain               4,282,881    191,633    0.04x
eye                   137,996    136,960    0.99x
heart                 178,606    170,554    0.95x
lungs                 534,154    530,143    0.99x
GI                    383,685    334,028    0.87x
liver               2,183,804    112,144    0.05x
spleen                 73,773     73,515    1.00x
pancreas               25,421     25,373    1.00x
kidney                111,204    111,062    1.00x
mesokidney             50,315     49,465    0.98x
collagen            8,528,692    713,108    0.08x
ear                    28,860     28,409    0.98x
nontissue             786,130     69,811    0.09x
thymus                 72,540     70,725    0.97x
thyroid                46,389     45,815    0.99x
bladder                17,079     16,272    0.95x
sk

---
## 8 · Train / val / test split

In [8]:
# ── Slide-level train / val / test split ──────────────────────────────────────
tiles_by_slide = defaultdict(list)
for tile in selected_tiles:
    tiles_by_slide[tile["slide_id"]].append(tile)

slide_ids    = list(tiles_by_slide.keys())
total_count  = len(selected_tiles)
target_train = int(total_count * TRAIN_RATIO)
target_val   = int(total_count * VAL_RATIO)

# Phase 1: ensure rare-class slides appear in all splits.
# Test is capped at MAX_TEST_SLIDES; excess slides redirect to train.
class_to_slides = defaultdict(list)
for sid, tiles in tiles_by_slide.items():
    for tile in tiles:
        for cls_id in tile["class_counts"]:
            class_to_slides[cls_id].append(sid)
class_to_slides = {k: list(set(v)) for k, v in class_to_slides.items()}

np.random.seed(SEED)
train_slides, val_slides, test_slides = set(), set(), set()

for cls_id in sorted(RARE_IDS):
    slides = [s for s in class_to_slides.get(cls_id, [])
              if s not in train_slides | val_slides | test_slides]
    if len(slides) >= 3:
        slides.sort(key=lambda s: len(tiles_by_slide[s]))
        train_slides.add(slides[0])
        val_slides.add(slides[len(slides) // 2])
        if len(test_slides) < MAX_TEST_SLIDES:
            test_slides.add(slides[-1])
        else:
            train_slides.add(slides[-1])   # redirect to train once test is full
    elif len(slides) == 2:
        train_slides.add(slides[0]); val_slides.add(slides[1])
    elif len(slides) == 1:
        train_slides.add(slides[0])

# Phase 2: greedily assign remaining slides to balance tile counts.
# Once test has MAX_TEST_SLIDES slides, only assign to train or val.
remaining = sorted(
    [s for s in slide_ids if s not in train_slides | val_slides | test_slides],
    key=lambda s: len(tiles_by_slide[s]), reverse=True
)
for sid in remaining:
    tr = sum(len(tiles_by_slide[s]) for s in train_slides)
    va = sum(len(tiles_by_slide[s]) for s in val_slides)
    te = sum(len(tiles_by_slide[s]) for s in test_slides)
    test_full = len(test_slides) >= MAX_TEST_SLIDES
    if test_full:
        if target_train - tr >= target_val - va:
            train_slides.add(sid)
        else:
            val_slides.add(sid)
    elif target_train - tr >= max(target_val - va, total_count - target_train - target_val - te):
        train_slides.add(sid)
    elif target_val - va >= total_count - target_train - target_val - te:
        val_slides.add(sid)
    else:
        test_slides.add(sid)

train_tiles_unique = [t for t in selected_tiles if t["slide_id"] in train_slides]
val_tiles          = [t for t in selected_tiles if t["slide_id"] in val_slides]
test_tiles         = [t for t in selected_tiles if t["slide_id"] in test_slides]

print(f"TRAIN : {len(train_slides):2d} slides -> {len(train_tiles_unique):,} tiles ({len(train_tiles_unique)/total_count*100:.1f}%)")
print(f"VAL   : {len(val_slides):2d} slides -> {len(val_tiles):,} tiles ({len(val_tiles)/total_count*100:.1f}%)")
print(f"TEST  : {len(test_slides):2d} slides -> {len(test_tiles):,} tiles ({len(test_tiles)/total_count*100:.1f}%)")


TRAIN :  6 slides → 9,111 tiles (22.8%)
VAL   :  4 slides → 13,566 tiles (33.9%)
TEST  :  4 slides → 17,323 tiles (43.3%)


---
## 9 · Oversample rare classes (train only)

In [9]:
# ── Oversample rare classes in training split (augmented duplicates) ───────────
# Each rare class is oversampled until it has at least OVERSAMPLE_TARGET tiles.
# Copies get a _dup{k} suffix and a deterministic geometric augmentation.

OVERSAMPLE_TARGETS = {
    "kidney":     2500, "pancreas":   2000, "bladder":    2500,
    "spleen":     1500, "spleen2":    1200, "thyroid":    2000,
    "ear":        1500, "eye":        1200, "mesokidney": 1500, "thymus": 1500,
}

train_by_cls = defaultdict(list)
for tile in train_tiles_unique:
    for cls_id in tile["class_counts"]:
        train_by_cls[cls_id].append(tile)

oversampled = []
np.random.seed(SEED + 1)

print(f"{'Class':<13} {'Have':>8} {'Target':>8} {'Added':>8}")
print("-" * 45)
for cls_name, tgt in OVERSAMPLE_TARGETS.items():
    cls_id    = name_to_id[cls_name]
    available = train_by_cls.get(cls_id, [])
    n_have    = len(available)
    if n_have == 0 or n_have >= tgt:
        print(f"  {cls_name:<11} {n_have:>8,} {tgt:>8,} {'—':>8}")
        continue
    tgt     = min(tgt, n_have * 8)  # cap at 8 augmentation variants
    n_add   = tgt - n_have
    chosen  = np.random.choice(n_have, size=n_add, replace=True)
    dup_ctr = {}
    for orig_idx in chosen:
        k = dup_ctr.get(orig_idx, 0) + 1
        dup_ctr[orig_idx] = k
        dup = dict(available[orig_idx])
        dup["dup_suffix"] = f"_dup{k}"
        oversampled.append(dup)
    print(f"  {cls_name:<11} {n_have:>8,} {tgt:>8,} {n_add:>8,}")

train_tiles = train_tiles_unique + oversampled

for t in train_tiles: t["split"] = "train"
for t in val_tiles:   t["split"] = "val"
for t in test_tiles:  t["split"] = "test"

print(f"\nFinal: TRAIN {len(train_tiles):,}  VAL {len(val_tiles):,}  TEST {len(test_tiles):,}")

Class             Have   Target    Added
---------------------------------------------
  kidney            88      704      616
  pancreas          10       80       70
  bladder          290    2,320    2,030
  spleen             6       48       42
  spleen2            5       40       35
  thyroid          363    2,000    1,637
  ear              101      808      707
  eye              789    1,200      411
  mesokidney       152    1,216    1,064
  thymus           400    1,500    1,100

Final: TRAIN 16,823  VAL 13,566  TEST 17,323


---
## 10 · Extract tiles — save PNGs and CSV labels

In [10]:
# ── Create output directory structure ─────────────────────────────────────────
# Per split, each tile produces three files:
#   images/tile_id.png                — RGB tile (256×256)
#   geojsons/tile_id.geojson          — localized polygon GeoJSON with class labels
#   labels/tile_id.png                — uint16 instance mask (0 = bg, 1…N = nuclei)
#   labels/tile_id_inst2class.json    — {"1": "kidney", "2": "bone", ...}
#
# The training pipeline (dataset_v2.py) reads images/ + labels/ directly.
# geojsons/ is kept for inspection in QuPath and for reprocessing if needed.

for split in ("train", "val", "test"):
    for sub in ("images", "geojsons", "labels"):
        (OUT_BASE / split / sub).mkdir(parents=True, exist_ok=True)

splits_dir = OUT_BASE / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

(OUT_BASE / "label_map.json").write_text(
    json.dumps({str(k): v for k, v in id_to_name.items()}, indent=2), encoding="utf-8"
)
print(f"Output directory: {OUT_BASE}")

# ── Extract PNG tiles + GeoJSON labels + instance masks ───────────────────────
all_output = train_tiles + val_tiles + test_tiles
by_slide   = defaultdict(list)
for tile in all_output:
    by_slide[tile["slide_id"]].append(tile)

print(f"Processing {len(by_slide)} slides, {len(all_output):,} tiles total")

tile_records = []
failed       = []

for slide_id, slide_tiles in tqdm(by_slide.items(), desc="Slides"):
    cells_xy     = slide_data[slide_id]["cells_xy"]
    slide_feats  = slide_data[slide_id]["features"]
    split_name   = slide_tiles[0]["split"]   # all tiles for a slide share the same split

    try:
        slide = openslide.OpenSlide(slide_data[slide_id]["image_path"])
    except Exception as exc:
        print(f"  ERROR opening {slide_id}: {exc}")
        failed += [(f"{slide_id}_{t['tile_x']}_{t['tile_y']}", str(exc)) for t in slide_tiles]
        continue

    for tile in slide_tiles:
        split      = tile["split"]
        dup_suffix = tile.get("dup_suffix", "")
        tile_id    = f"{slide_id}_x{tile['tile_x']}_y{tile['tile_y']}{dup_suffix}"
        m          = re.search(r"_dup(\d+)$", dup_suffix)
        aug_id     = (int(m.group(1)) % 8) if m else 0

        images_dir   = OUT_BASE / split / "images"
        geojsons_dir = OUT_BASE / split / "geojsons"
        labels_dir   = OUT_BASE / split / "labels"

        try:
            # ── 1. Read and augment RGB tile ──────────────────────────────────
            pil = slide.read_region((tile["tile_x"], tile["tile_y"]), 0, (TILE_SIZE, TILE_SIZE))
            img = np.array(pil.convert("RGB"))
            if img.shape[:2] != (TILE_SIZE, TILE_SIZE):
                img = cv2.resize(img, (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_LINEAR)
            img = augment_image(img, aug_id)

            cv2.imwrite(
                str(images_dir / f"{tile_id}.png"),
                cv2.cvtColor(img, cv2.COLOR_RGB2BGR),
            )

            # ── 2. Clip GeoJSON features to this tile and localize coordinates ─
            tile_feats = clip_features_to_tile(
                slide_feats,
                tile["tile_x"], tile["tile_y"], TILE_SIZE,
                tile["cell_indices"],
            )

            # ── 3. Augment polygon coordinates to match the image augmentation ─
            if aug_id != 0:
                for feat in tile_feats:
                    rings = feat["geometry"]["coordinates"]
                    feat["geometry"]["coordinates"] = [
                        augment_polygon_ring(ring, aug_id, TILE_SIZE)
                        for ring in rings
                    ]

            # ── 4. Write per-tile GeoJSON (QuPath-compatible, for inspection) ──
            write_tile_geojson(tile_feats, geojsons_dir / f"{tile_id}.geojson")

            # ── 5. Rasterize polygons → uint16 instance mask + inst2class.json ─
            mask, inst2class = rasterize_tile_features(tile_feats, TILE_SIZE)
            cv2.imwrite(str(labels_dir / f"{tile_id}.png"), mask)
            (labels_dir / f"{tile_id}_inst2class.json").write_text(
                json.dumps(inst2class), encoding="utf-8"
            )

            tile_records.append({
                "tile_id":      tile_id,
                "slide_id":     slide_id,
                "split":        split,
                "num_cells":    tile["num_cells"],
                "is_duplicate": bool(dup_suffix),
                "aug_id":       aug_id,
            })
        except Exception as exc:
            failed.append((tile_id, str(exc)))

    slide.close()

n_ok  = len(tile_records)
n_dup = sum(1 for r in tile_records if r["is_duplicate"])
print(f"\n[DONE] {n_ok:,} tiles saved  ({n_ok-n_dup:,} unique + {n_dup:,} augmented duplicates)")
if failed:
    print(f"Failed: {len(failed)}")
    for tid, err in failed[:5]:
        print(f"  {tid}: {err}")

Output directory: \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_40k_GS55
Processing 14 slides, 47,712 tiles total


Slides:   0%|          | 0/14 [00:00<?, ?it/s]


[DONE] 47,712 tiles saved  (40,000 unique + 7,712 augmented duplicates)


---
## 11 · Write split files and training config

In [11]:
# ── Write split CSV files and training config ──────────────────────────────────
df_tiles  = pd.DataFrame(tile_records)

train_ids = df_tiles[df_tiles["split"] == "train"]["tile_id"].tolist()
val_ids   = df_tiles[df_tiles["split"] == "val"]["tile_id"].tolist()
test_ids  = df_tiles[df_tiles["split"] == "test"]["tile_id"].tolist()

fold_dir = splits_dir / "fold_0"
fold_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame(train_ids).to_csv(fold_dir / "train.csv", index=False, header=False)
pd.DataFrame(val_ids).to_csv(fold_dir / "val.csv",     index=False, header=False)
pd.DataFrame(test_ids).to_csv(splits_dir / "test.csv", index=False, header=False)

# ── Training config for dataset_v2 / train_v2 ─────────────────────────────────
# The pipeline reads images/ and labels/ directly.
# labels/ contains:  tile_id.png (uint16 instance mask)
#                    tile_id_inst2class.json (instance → class name)
config = {
    "logging": {
        "mode": "online",
        "project": "convnext_stardist_GS55",
        "log_comment": "GS55_multitask_20x",
    },
    "data": {
        "train_images_dir": str(OUT_BASE / "train" / "images"),
        "train_labels_dir": str(OUT_BASE / "train" / "labels"),
        "val_images_dir":   str(OUT_BASE / "val"   / "images"),
        "val_labels_dir":   str(OUT_BASE / "val"   / "labels"),
        "train_stems":      str(fold_dir / "train.csv"),
        "val_stems":        str(fold_dir / "val.csv"),
        "num_classes":      len(LABELS),
        "label_map":        id_to_name,
    },
    "model": {
        "class_names": LABELS,
        "num_classes": len(LABELS),
        "n_rays": 32,
    },
    "train": {
        "patch_size":    TILE_SIZE,
        "batch_size":    8,
        "epochs":        50,
        "learning_rate": 0.0001,
        "loss_w_cls":    2.0,
        "loss_w_inst":   0.5,
        "loss_w_dist":   0.05,
        "class_weights": "auto",
    },
}

config_dir  = OUT_BASE / "train_configs"
config_dir.mkdir(parents=True, exist_ok=True)
config_path = config_dir / "train_config.yaml"
config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

(OUT_BASE / "label_map.yaml").write_text(
    yaml.safe_dump({"labels": id_to_name}), encoding="utf-8"
)

print(f"TRAIN: {len(train_ids):,} tiles  VAL: {len(val_ids):,}  TEST: {len(test_ids):,}")
print(f"Splits → {splits_dir}")
print(f"Config → {config_path}")
print("\nDataset layout:")
for split in ("train", "val", "test"):
    imgs = list((OUT_BASE / split / "images").glob("*.png"))
    masks = list((OUT_BASE / split / "labels").glob("*.png"))
    print(f"  {split:5s}: {len(imgs):,} images  {len(masks):,} instance masks")
print("\nDataset creation complete!")

TRAIN: 16,823 tiles  VAL: 13,566  TEST: 17,323
Splits → \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_40k_GS55\splits
Config → \\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS55\cellvit_training\dataset_256_40k_GS55\train_configs\train_config.yaml

Dataset layout:
  train: 16,384 images  16,384 instance masks
  val  : 13,566 images  13,566 instance masks
  test : 17,323 images  17,323 instance masks

Dataset creation complete!


In [ ]:
# ── Organ coverage summary: tiles per class per split ─────────────────────────
from collections import defaultdict as _dd

split_cls_tiles = {"train": _dd(int), "val": _dd(int), "test": _dd(int)}

for tile in train_tiles:
    for cls_id in tile["class_counts"]:
        split_cls_tiles["train"][id_to_name[cls_id]] += 1

for tile in val_tiles:
    for cls_id in tile["class_counts"]:
        split_cls_tiles["val"][id_to_name[cls_id]] += 1

for tile in test_tiles:
    for cls_id in tile["class_counts"]:
        split_cls_tiles["test"][id_to_name[cls_id]] += 1

print(f"{'Organ':<14} {'TRAIN':>8} {'VAL':>8} {'TEST':>8}  Coverage")
print("-" * 60)
for name in LABELS:
    tr = split_cls_tiles["train"][name]
    va = split_cls_tiles["val"][name]
    te = split_cls_tiles["test"][name]
    flags = []
    if tr == 0: flags.append("NO TRAIN")
    if va == 0: flags.append("NO VAL")
    if te == 0: flags.append("NO TEST")
    flag_str = "  WARNING: " + ", ".join(flags) if flags else ""
    print(f"{name:<14} {tr:>8,} {va:>8,} {te:>8,}{flag_str}")

print(f"\n{'TOTAL':<14} {len(train_tiles):>8,} {len(val_tiles):>8,} {len(test_tiles):>8,}")
